In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import pickle
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_curve, auc, 
    confusion_matrix, ConfusionMatrixDisplay
)

# ═══════════════════════════════════════════════════════════════
# 1. CONFIGURATION & LOADING
# ═══════════════════════════════════════════════════════════════
# Adjust these paths to your Kaggle environment
INPUT_DIR = Path("/kaggle/input/datasets/nisarggandhi22/train-ready") 
OUTPUT_DIR = Path("/kaggle/working")

print("Loading datasets...")
df1 = pl.read_parquet(INPUT_DIR / "plant_1_train_ready.parquet")
df2 = pl.read_parquet(INPUT_DIR / "plant_2_train_ready.parquet")
df3 = pl.read_parquet(INPUT_DIR / "plant_3_train_ready.parquet")

# ═══════════════════════════════════════════════════════════════
# 2. FEATURE SELECTION & COMBINATION
# ═══════════════════════════════════════════════════════════════
# Define the Top 20 Features identified via SHAP analysis
TOP_20_FEATURES = [
    "inv_kwh_total", "roll_temp_mean_7d", "roll_kwh_today_std_7d", 
    "roll_temp_std_7d", "roll_temp_std_3d", "roll_temp_mean_3d", 
    "roll_pv1_power_std_7d", "anom_night_power_7d", "roll_kwh_today_mean_7d", 
    "roll_kwh_today_std_3d", "str_worst_ratio_rmean_7d", "day_of_week", 
    "inv_power", "str_mean_rmean_7d", "roll_kwh_today_mean_3d", 
    "stress_hightemp_7d", "is_daytime", "anom_night_hightemp_7d", 
    "roll_power_std_3d", "roll_pv1_power_mean_3d"
]

# Columns needed for tracking and target
METADATA_COLS = ["plant_id", "target"]

# Combine the plants using only the necessary columns
all_data = []
for i, df in enumerate([df1, df2, df3], 1):
    # Select only top features + metadata that exist in the dataframe
    available_cols = [c for c in TOP_20_FEATURES + METADATA_COLS if c in df.columns]
    pdf = df.select(available_cols).to_pandas()
    all_data.append(pdf)

combined_df = pd.concat(all_data, ignore_index=True)

# One-hot encode plant_id so the model understands site differences
combined_df = pd.get_dummies(combined_df, columns=['plant_id'], drop_first=False)

# ═══════════════════════════════════════════════════════════════
# 3. DATA SPLITTING (70/15/15)
# ═══════════════════════════════════════════════════════════════
X = combined_df.drop(columns=["target"])
y = combined_df["target"]

# Split into 70% Train, 30% Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Split Temp into 15% Val and 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")

# ═══════════════════════════════════════════════════════════════
# 4. MODEL TRAINING
# ═══════════════════════════════════════════════════════════════
scale_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_weight,
    eval_metric="aucpr",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

print("\nStarting training...")
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

# ═══════════════════════════════════════════════════════════════
# 5. EVALUATION (THRESHOLD = 0.60)
# ═══════════════════════════════════════════════════════════════
CUSTOM_THRESHOLD = 0.60
y_probs = model.predict_proba(X_test)[:, 1]
y_pred = (y_probs >= CUSTOM_THRESHOLD).astype(int)

print("\n" + "="*40)
print(f"FINAL TEST SET RESULTS (Threshold: {CUSTOM_THRESHOLD})")
print("="*40)
print(classification_report(y_test, y_pred, target_names=["Healthy (0)", "Failure (1)"]))

# Numerical ROC AUC
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
print(f"AUC ROC Score: {auc(fpr, tpr):.4f}")

# ═══════════════════════════════════════════════════════════════
# 6. SAVE MODEL AND ASSETS
# ═══════════════════════════════════════════════════════════════
# Save .pkl model
with open(OUTPUT_DIR / "xgboost_solar_model.pkl", 'wb') as f:
    pickle.dump(model, f)

# Save feature list for deployment consistency
with open(OUTPUT_DIR / "feature_list.json", 'w') as f:
    json.dump(list(X_train.columns), f)

# ═══════════════════════════════════════════════════════════════
# 7. SHAP EXPLAINABILITY ANALYSIS (Hackathon XAI Requirement)
# ═══════════════════════════════════════════════════════════════
print("\n" + "="*40)
print("SHAP EXPLAINABILITY ANALYSIS")
print("="*40)

# Use a sample of the test set for faster SHAP calculation (max 5000 rows)
X_sample = X_test.sample(n=min(5000, len(X_test)), random_state=42)

print("Calculating SHAP values...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# 1. Calculate and Print Numerical Feature Importance
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Mean_Absolute_SHAP': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean_Absolute_SHAP', ascending=False)

print("\n--- TOP 10 PREDICTORS OF FAILURE ---")
print(importance_df.head(10).to_string(index=False))

# 2. Save and Show the SHAP Summary Plot
plt.figure(figsize=(10, 8))
# Ensure you are passing the DataFrame so column names appear on the plot
shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
plt.title("SolarSight: Top 20 Features (SHAP Importance)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shap_summary_plot.png")
plt.show()

# 3. Export importance to CSV for your presentation/report
importance_df.to_csv(OUTPUT_DIR / "feature_importance_values.csv", index=False)

print(f"\n✅ SHAP Summary Plot and CSV saved to {OUTPUT_DIR}")